# AWS HealthOmics SDK Integration Test
## HealthOmics Demo - VPC Private Endpoint Validation

This notebook validates:
1. **VPC Connectivity** - SDK calls route through private VPC endpoints
2. **Workflow Management** - List, describe, and run workflows
3. **Storage Operations** - Sequence stores, reference stores
4. **Analytics** - Variant stores, annotation stores
5. **Security Validation** - IAM permissions, encryption, audit logging

### References
- [VPC Interface Endpoints](https://docs.aws.amazon.com/omics/latest/dev/vpc-interface-endpoints.html)
- [IAM with HealthOmics](https://docs.aws.amazon.com/omics/latest/dev/security_iam_service-with-iam.html)
- [Data Protection](https://docs.aws.amazon.com/omics/latest/dev/data-protection.html)
- [Confused Deputy Prevention](https://docs.aws.amazon.com/omics/latest/dev/cross-service-confused-deputy-prevention.html)

---
## 0. Environment Setup

In [ ]:
import boto3
import json
import time
from datetime import datetime, timezone
from botocore.config import Config

REGION = 'us-west-2'
ACCOUNT_ID = '<AWS_ACCOUNT_ID>'

# Configure boto3 with retry and timeout settings
config = Config(
    region_name=REGION,
    retries={'max_attempts': 3, 'mode': 'adaptive'},
    connect_timeout=5,
    read_timeout=30
)

# Initialize clients
omics = boto3.client('omics', config=config)
sts = boto3.client('sts', config=config)
ec2 = boto3.client('ec2', config=config)
cloudtrail = boto3.client('cloudtrail', config=config)

# Verify identity
identity = sts.get_caller_identity()
print(f"Account:  {identity['Account']}")
print(f"Role ARN: {identity['Arn']}")
print(f"Region:   {REGION}")
print(f"Time:     {datetime.now(timezone.utc).isoformat()}")

---
## 1. VPC Connectivity Validation
Verify that all HealthOmics API calls route through VPC private endpoints (no public internet).

In [ ]:
import socket

# HealthOmics VPC Endpoint DNS names to verify
endpoints = [
    'workflows-omics.us-west-2.amazonaws.com',
    'storage-omics.us-west-2.amazonaws.com',
    'analytics-omics.us-west-2.amazonaws.com',
    'tags-omics.us-west-2.amazonaws.com',
    'control-storage-omics.us-west-2.amazonaws.com',
]

print("=" * 70)
print("VPC Endpoint DNS Resolution Test")
print("Private IPs (10.x.x.x) confirm traffic routes through VPC endpoints")
print("=" * 70)

all_private = True
for ep in endpoints:
    try:
        ip = socket.gethostbyname(ep)
        is_private = ip.startswith('10.')
        status = 'PASS (Private)' if is_private else 'WARN (Public)'
        if not is_private:
            all_private = False
        print(f"  {ep}")
        print(f"    -> {ip}  [{status}]")
    except socket.gaierror as e:
        print(f"  {ep}")
        print(f"    -> FAIL: {e}")
        all_private = False

print("\n" + "=" * 70)
print(f"Result: {'ALL ENDPOINTS PRIVATE - VPC routing confirmed' if all_private else 'SOME ENDPOINTS NOT PRIVATE - check VPC endpoint config'}")
print("=" * 70)

In [ ]:
# Verify VPC Endpoints status from EC2 API
vpc_id = '<VPC_ID>'

response = ec2.describe_vpc_endpoints(
    Filters=[{'Name': 'vpc-id', 'Values': [vpc_id]}]
)

print(f"{'Service':<55} {'State':<12} {'Type'}")
print("-" * 85)
for ep in sorted(response['VpcEndpoints'], key=lambda x: x['ServiceName']):
    svc = ep['ServiceName'].replace(f'com.amazonaws.{REGION}.', '')
    print(f"  {svc:<53} {ep['State']:<12} {ep['VpcEndpointType']}")

omics_eps = [ep for ep in response['VpcEndpoints'] if 'omics' in ep['ServiceName']]
print(f"\nHealthOmics endpoints: {len(omics_eps)}/5 configured")
print(f"All available: {all(ep['State'] == 'available' for ep in omics_eps)}")

---
## 2. Workflow Management
Test listing, describing, and monitoring HealthOmics workflows.

In [ ]:
# 2.1 List all workflows
print("=" * 70)
print("Available HealthOmics Workflows")
print("=" * 70)

workflows = []
paginator = omics.get_paginator('list_workflows')
for page in paginator.paginate():
    workflows.extend(page.get('items', []))

print(f"\n{'ID':<12} {'Name':<35} {'Status':<10} {'Type':<10} {'Created'}")
print("-" * 95)
for wf in workflows:
    created = wf.get('creationTime', '').strftime('%Y-%m-%d') if hasattr(wf.get('creationTime', ''), 'strftime') else str(wf.get('creationTime', ''))[:10]
    print(f"  {wf['id']:<10} {wf.get('name','N/A'):<35} {wf['status']:<10} {wf.get('type','N/A'):<10} {created}")

print(f"\nTotal workflows: {len(workflows)}")

In [ ]:
# 2.2 Describe a specific workflow in detail
WORKFLOW_ID = '<WORKFLOW_ID_1>'  # nf-core/scrnaseq

wf_detail = omics.get_workflow(id=WORKFLOW_ID)

print(f"Workflow: {wf_detail.get('name')}")
print(f"  ID:          {wf_detail['id']}")
print(f"  ARN:         {wf_detail['arn']}")
print(f"  Status:      {wf_detail['status']}")
print(f"  Type:        {wf_detail.get('type')}")
print(f"  Engine:      {wf_detail.get('engine', 'N/A')}")
print(f"  Storage (GB): {wf_detail.get('storageCapacity', 'N/A')}")
print(f"  Created:     {wf_detail.get('creationTime')}")
print(f"  Digest:      {wf_detail.get('digest', 'N/A')}")

# Show parameter template if available
params = wf_detail.get('parameterTemplate', {})
if params:
    print(f"\n  Parameters ({len(params)}):")
    for name, spec in list(params.items())[:10]:
        optional = spec.get('optional', False)
        desc = spec.get('description', '')
        print(f"    - {name}: {desc} {'(optional)' if optional else '(required)'}")

In [ ]:
# 2.3 List workflow runs
print("=" * 70)
print("Recent Workflow Runs")
print("=" * 70)

runs = []
try:
    paginator = omics.get_paginator('list_runs')
    for page in paginator.paginate(PaginationConfig={'MaxItems': 20}):
        runs.extend(page.get('items', []))
except Exception as e:
    print(f"Note: {e}")

if runs:
    print(f"\n{'Run ID':<15} {'Status':<15} {'Workflow ID':<12} {'Created'}")
    print("-" * 65)
    for run in runs[:10]:
        created = str(run.get('creationTime', ''))[:19]
        print(f"  {run['id']:<13} {run.get('status','N/A'):<15} {run.get('workflowId','N/A'):<12} {created}")
    print(f"\nTotal runs found: {len(runs)}")
else:
    print("\nNo workflow runs found.")

---
## 3. Storage Operations
Test Sequence Store and Reference Store access.

In [ ]:
# 3.1 List and describe Sequence Stores
print("=" * 70)
print("Sequence Stores")
print("=" * 70)

seq_stores = omics.list_sequence_stores()
for store in seq_stores.get('sequenceStores', []):
    print(f"\n  Name:      {store['name']}")
    print(f"  ID:        {store['id']}")
    print(f"  ARN:       {store['arn']}")
    print(f"  Status:    {store.get('status', 'N/A')}")
    print(f"  Created:   {store['creationTime']}")
    print(f"  Fallback:  {store.get('fallbackLocation', 'N/A')}")
    
    # List read sets in this store
    SEQUENCE_STORE_ID = store['id']
    try:
        read_sets = omics.list_read_sets(sequenceStoreId=SEQUENCE_STORE_ID)
        rs_items = read_sets.get('readSets', [])
        print(f"  Read Sets: {len(rs_items)}")
        for rs in rs_items[:5]:
            print(f"    - {rs['id']}: {rs.get('name','N/A')} ({rs.get('fileType','N/A')}, {rs.get('status','N/A')})")
    except Exception as e:
        print(f"  Read Sets: Error - {e}")

In [ ]:
# 3.2 List and describe Reference Stores
print("=" * 70)
print("Reference Stores")
print("=" * 70)

ref_stores = omics.list_reference_stores()
for store in ref_stores.get('referenceStores', []):
    print(f"\n  Name:    {store['name']}")
    print(f"  ID:      {store['id']}")
    print(f"  ARN:     {store['arn']}")
    print(f"  Created: {store['creationTime']}")
    
    REFERENCE_STORE_ID = store['id']
    # List references
    try:
        refs = omics.list_references(referenceStoreId=REFERENCE_STORE_ID)
        ref_items = refs.get('references', [])
        print(f"  References: {len(ref_items)}")
        for ref in ref_items[:5]:
            print(f"    - {ref['id']}: {ref.get('name','N/A')} ({ref.get('status','N/A')})")
    except Exception as e:
        print(f"  References: Error - {e}")

---
## 4. Analytics Operations
Test Variant Store and Annotation Store access.

In [ ]:
# 4.1 Variant Stores
print("=" * 70)
print("Variant Stores")
print("=" * 70)

variant_stores = omics.list_variant_stores()
for vs in variant_stores.get('variantStores', []):
    size_mb = vs.get('storeSizeBytes', 0) / (1024 * 1024)
    print(f"\n  Name:      {vs['name']}")
    print(f"  ID:        {vs['id']}")
    print(f"  ARN:       {vs['storeArn']}")
    print(f"  Status:    {vs['status']}")
    print(f"  Size:      {size_mb:.1f} MB")
    print(f"  Reference: {vs.get('reference', {}).get('referenceArn', 'N/A')}")
    print(f"  Created:   {vs['creationTime']}")

In [ ]:
# 4.2 Annotation Stores
print("=" * 70)
print("Annotation Stores")
print("=" * 70)

annotation_stores = omics.list_annotation_stores()
for ann in annotation_stores.get('annotationStores', []):
    size_mb = ann.get('storeSizeBytes', 0) / (1024 * 1024)
    print(f"\n  Name:      {ann['name']}")
    print(f"  ID:        {ann['id']}")
    print(f"  ARN:       {ann['storeArn']}")
    print(f"  Status:    {ann['status']}")
    print(f"  Format:    {ann.get('storeFormat', 'N/A')}")
    print(f"  Size:      {size_mb:.1f} MB")
    print(f"  Reference: {ann.get('reference', {}).get('referenceArn', 'N/A')}")
    print(f"  Created:   {ann['creationTime']}")

In [ ]:
# 4.3 Query Variant Store via Athena (if Lake Formation integration exists)
VARIANT_STORE_NAME = 'omicsvariantstore'

try:
    vs_detail = omics.get_variant_store(name=VARIANT_STORE_NAME)
    print(f"Variant Store Detail: {VARIANT_STORE_NAME}")
    print(f"  Status:      {vs_detail['status']}")
    print(f"  Size:        {vs_detail.get('storeSizeBytes', 0) / (1024*1024):.1f} MB")
    print(f"  Description: {vs_detail.get('description', 'N/A')}")
    print(f"  Updated:     {vs_detail.get('updateTime', 'N/A')}")
except Exception as e:
    print(f"Error: {e}")

---
## 5. Workflow Execution Test
Start a workflow run and monitor its progress.

### Available Workflows & Required Parameters

| Workflow ID | Name | Engine | Required Parameters |
|-------------|------|--------|-------------------|
| <WORKFLOW_ID_3> | perf-test-wdl-us-west-2 | WDL | `docker_image` (Docker image URI) |
| <WORKFLOW_ID_2> | perf-test-nextflow-us-west-2 | NEXTFLOW | `docker_image` (Docker image URI) |
| <WORKFLOW_ID_4> | EvoProtGrad | NEXTFLOW | `container_image` (ECR URI), `seed_sequence` (protein seq) |
| <WORKFLOW_ID_1> | nf-core/scrnaseq | NEXTFLOW | `input` (samplesheet CSV), `outdir` (S3 output path) |

### Known Working Configuration (from previous runs)
- **docker_image**: `<AWS_ACCOUNT_ID>.dkr.ecr.us-west-2.amazonaws.com/<S3_OUTPUT_BUCKET_PREFIX>:latest`
- **roleArn**: `arn:aws:iam::<AWS_ACCOUNT_ID>:role/omics-run-role`
- **outputUri**: `s3://<S3_OUTPUT_BUCKET>/outputs/`

In [ ]:
# 5.1 Inspect workflow parameters before running

# Use the Nextflow perf-test workflow (previously completed successfully as run <PREVIOUS_RUN_ID>)
WORKFLOW_ID_TO_RUN = '<WORKFLOW_ID_2>'  # perf-test-nextflow-us-west-2 (COMPLETED in previous run)

# Known working values from run <PREVIOUS_RUN_ID> (COMPLETED 2026-03-07)
ECR_IMAGE = '<AWS_ACCOUNT_ID>.dkr.ecr.us-west-2.amazonaws.com/<S3_OUTPUT_BUCKET_PREFIX>:latest'
OMICS_SERVICE_ROLE = f'arn:aws:iam::{ACCOUNT_ID}:role/omics-run-role'
OUTPUT_URI = f's3://<S3_OUTPUT_BUCKET_PREFIX>-{REGION}-{ACCOUNT_ID}/outputs/'

# Get workflow details and verify parameters
wf = omics.get_workflow(id=WORKFLOW_ID_TO_RUN)
print(f"Workflow: {wf.get('name')}")
print(f"Engine:   {wf.get('engine', 'N/A')}")
print(f"Status:   {wf['status']}")

print(f"\nRequired Parameters:")
for name, spec in wf.get('parameterTemplate', {}).items():
    optional = spec.get('optional', False)
    desc = spec.get('description', '')
    print(f"  {name}: {desc} {'(optional)' if optional else '(REQUIRED)'}")

print(f"\nPlanned Run Configuration:")
print(f"  docker_image: {ECR_IMAGE}")
print(f"  roleArn:      {OMICS_SERVICE_ROLE}")
print(f"  outputUri:     {OUTPUT_URI}")
print(f"\nNote: WDL workflow (<WORKFLOW_ID_3>) uses public.ecr.aws internally and fails.")
print(f"      This Nextflow workflow (<WORKFLOW_ID_2>) was verified in run <PREVIOUS_RUN_ID>.")

In [ ]:
# 5.2 Execute workflow run
run_name = f'bioepis-test-{datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")}'

run_response = omics.start_run(
    workflowId=WORKFLOW_ID_TO_RUN,
    roleArn=OMICS_SERVICE_ROLE,
    outputUri=OUTPUT_URI,
    name=run_name,
    parameters={
        'docker_image': ECR_IMAGE,
    },
    storageType='DYNAMIC',
    tags={
        'Project': 'HealthOmicsDemo-Demo',
        'Environment': 'Test'
    }
)

RUN_ID = run_response['id']
print(f"Run started successfully!")
print(f"  Run ID:  {RUN_ID}")
print(f"  Name:    {run_name}")
print(f"  ARN:     {run_response['arn']}")
print(f"  Status:  {run_response['status']}")

In [ ]:
# 5.3 Monitor the workflow run
def monitor_run(run_id, poll_interval=30, max_polls=20):
    """Monitor a workflow run until completion or timeout."""
    terminal_states = {'COMPLETED', 'FAILED', 'CANCELLED', 'DELETED'}
    
    for i in range(max_polls):
        run = omics.get_run(id=run_id)
        status = run['status']
        
        print(f"[{datetime.now(timezone.utc).strftime('%H:%M:%S')}] Run {run_id}: {status}")
        
        if status in terminal_states:
            print(f"\nFinal Status: {status}")
            if status == 'COMPLETED':
                print(f"Output URI: {run.get('outputUri', 'N/A')}")
                start = run.get('startTime')
                stop = run.get('stopTime')
                if start and stop:
                    duration = (stop - start).total_seconds()
                    print(f"Duration: {duration:.0f}s ({duration/60:.1f}m)")
            elif status == 'FAILED':
                print(f"Failure Reason: {run.get('statusMessage', 'N/A')}")
            return run
        
        time.sleep(poll_interval)
    
    print(f"Monitoring timeout after {max_polls * poll_interval}s")
    return None

# Monitor the run started in 5.2
monitor_run(RUN_ID)

---
## 6. Resource Management
Test create/update operations for HealthOmics resources.

In [ ]:
# 6.1 Create a Run Group for resource management

try:
    rg = omics.create_run_group(
        name='bioepis-test-run-group',
        maxCpus=16,
        maxRuns=5,
        maxDuration=3600,
        tags={
            'Project': 'HealthOmicsDemo-Demo',
            'Environment': 'Test'
        }
    )
    print(f"Run Group created:")
    print(f"  ID:   {rg['id']}")
    print(f"  ARN:  {rg['arn']}")
    print(f"  Tags: {rg.get('tags', {})}")
    RUN_GROUP_ID = rg['id']
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# 6.2 Tag management - list and add tags to resources

# List tags on a workflow
WORKFLOW_ARN = f'arn:aws:omics:{REGION}:{ACCOUNT_ID}:workflow/<WORKFLOW_ID_1>'

try:
    tags = omics.list_tags_for_resource(resourceArn=WORKFLOW_ARN)
    print(f"Tags on {WORKFLOW_ARN}:")
    for k, v in tags.get('tags', {}).items():
        print(f"  {k}: {v}")
    if not tags.get('tags'):
        print("  (no tags)")
except Exception as e:
    print(f"Error: {e}")

# Add a test tag
try:
    omics.tag_resource(
        resourceArn=WORKFLOW_ARN,
        tags={'BioEpis-Test': 'SDK-Validation', 'TestDate': datetime.utcnow().strftime('%Y-%m-%d')}
    )
    print(f"\nTags added successfully.")
    
    # Verify
    tags = omics.list_tags_for_resource(resourceArn=WORKFLOW_ARN)
    print("Updated tags:")
    for k, v in tags.get('tags', {}).items():
        print(f"  {k}: {v}")
except Exception as e:
    print(f"Error adding tags: {e}")

---
## 7. Security Validation

In [ ]:
# 7.1 Verify IAM Role permissions (should succeed)
print("=" * 70)
print("IAM Permission Validation")
print("=" * 70)

test_operations = [
    ('list_workflows', lambda: omics.list_workflows(maxResults=1)),
    ('list_runs', lambda: omics.list_runs(maxResults=1)),
    ('list_sequence_stores', lambda: omics.list_sequence_stores(maxResults=1)),
    ('list_reference_stores', lambda: omics.list_reference_stores(maxResults=1)),
    ('list_variant_stores', lambda: omics.list_variant_stores(maxResults=1)),
    ('list_annotation_stores', lambda: omics.list_annotation_stores(maxResults=1)),
    ('list_run_groups', lambda: omics.list_run_groups(maxResults=1)),
]

results = []
for name, op in test_operations:
    try:
        op()
        results.append((name, 'PASS'))
        print(f"  {name:<30} PASS")
    except Exception as e:
        error_code = getattr(e, 'response', {}).get('Error', {}).get('Code', str(e))
        results.append((name, f'FAIL ({error_code})'))
        print(f"  {name:<30} FAIL ({error_code})")

passed = sum(1 for _, r in results if r == 'PASS')
print(f"\nResult: {passed}/{len(results)} operations passed")

In [ ]:
# 7.2 Verify encryption settings
print("=" * 70)
print("Encryption Validation")
print("Ref: https://docs.aws.amazon.com/omics/latest/dev/data-protection.html")
print("=" * 70)

# Check Sequence Store encryption
SEQUENCE_STORE_ID = '<SEQUENCE_STORE_ID>'
try:
    store = omics.get_sequence_store(id=SEQUENCE_STORE_ID)
    sse = store.get('sseConfig', {})
    print(f"\nSequence Store: {store['name']}")
    print(f"  Encryption Type: {sse.get('type', 'AWS_OWNED_KMS_KEY (default)')}")
    print(f"  KMS Key ARN:     {sse.get('keyArn', 'AWS managed')}")
except Exception as e:
    print(f"Error: {e}")

# TLS verification
import ssl
import urllib.request

print(f"\nTLS Configuration:")
print(f"  OpenSSL Version:  {ssl.OPENSSL_VERSION}")
print(f"  Default Protocol: TLS 1.2+ (enforced by boto3)")
print(f"  Recommendation:   TLS 1.3 (per AWS docs)")

In [ ]:
# 7.3 CloudTrail audit log verification
print("=" * 70)
print("CloudTrail Audit Log Check")
print("=" * 70)

try:
    events = cloudtrail.lookup_events(
        LookupAttributes=[
            {'AttributeKey': 'EventSource', 'AttributeValue': 'omics.amazonaws.com'}
        ],
        MaxResults=10
    )
    
    ct_events = events.get('Events', [])
    if ct_events:
        print(f"\nRecent HealthOmics API calls ({len(ct_events)}):")
        print(f"{'Time':<22} {'Event':<35} {'User'}")
        print("-" * 80)
        for evt in ct_events:
            event_time = evt['EventTime'].strftime('%Y-%m-%d %H:%M:%S')
            print(f"  {event_time:<20} {evt['EventName']:<35} {evt.get('Username', 'N/A')}")
    else:
        print("\nNo recent HealthOmics events found in CloudTrail.")
        print("Note: CloudTrail events may take 5-15 minutes to appear.")
except Exception as e:
    print(f"Error querying CloudTrail: {e}")

---
## 8. Performance Benchmark
Measure SDK response times to verify SLA compliance (<5s).

In [ ]:
import statistics

print("=" * 70)
print("SDK Response Time Benchmark (via VPC Endpoints)")
print("SLA Target: < 5 seconds per API call")
print("=" * 70)

benchmarks = {
    'list_workflows': lambda: omics.list_workflows(maxResults=5),
    'list_runs': lambda: omics.list_runs(maxResults=5),
    'list_sequence_stores': lambda: omics.list_sequence_stores(maxResults=5),
    'list_variant_stores': lambda: omics.list_variant_stores(maxResults=5),
    'get_workflow': lambda: omics.get_workflow(id='<WORKFLOW_ID_1>'),
}

N_RUNS = 3  # runs per operation
all_times = []

print(f"\n{'Operation':<30} {'Avg (ms)':<12} {'Min (ms)':<12} {'Max (ms)':<12} {'Status'}")
print("-" * 85)

for name, op in benchmarks.items():
    times = []
    for _ in range(N_RUNS):
        start = time.time()
        try:
            op()
            elapsed = (time.time() - start) * 1000
            times.append(elapsed)
        except Exception:
            times.append(-1)
    
    valid = [t for t in times if t >= 0]
    if valid:
        avg = statistics.mean(valid)
        mn = min(valid)
        mx = max(valid)
        status = 'PASS' if mx < 5000 else 'FAIL (>5s)'
        all_times.extend(valid)
        print(f"  {name:<28} {avg:<12.0f} {mn:<12.0f} {mx:<12.0f} {status}")
    else:
        print(f"  {name:<28} {'ERROR':<12} {'-':<12} {'-':<12} FAIL")

if all_times:
    print(f"\nOverall: avg={statistics.mean(all_times):.0f}ms, "
          f"p50={sorted(all_times)[len(all_times)//2]:.0f}ms, "
          f"max={max(all_times):.0f}ms")
    print(f"SLA Compliance: {'PASS' if max(all_times) < 5000 else 'FAIL'}")

---
## 9. Cleanup (Optional)

In [ ]:
# Remove test tags
try:
    omics.untag_resource(
        resourceArn=WORKFLOW_ARN,
        tagKeys=['BioEpis-Test', 'TestDate']
    )
    print("Test tags removed from workflow.")
except Exception as e:
    print(f"Cleanup error: {e}")

# Delete test run group
try:
    if 'RUN_GROUP_ID' in dir():
        omics.delete_run_group(id=RUN_GROUP_ID)
        print(f"Run group {RUN_GROUP_ID} deleted.")
except Exception as e:
    print(f"Cleanup error: {e}")

print("\nCleanup complete.")

---
## Test Summary

| Test Area | Cells | Description |
|-----------|-------|-------------|
| VPC Connectivity | 1, 2 | DNS resolution confirms private endpoint routing |
| Workflow Mgmt | 3-5 | List, describe, run workflows via SDK |
| Storage Ops | 6-7 | Sequence stores, reference stores, read sets |
| Analytics | 8-10 | Variant stores, annotation stores |
| Workflow Exec | 11-13 | Start and monitor workflow runs |
| Resource Mgmt | 14-15 | Run groups, tag management |
| Security | 16-18 | IAM validation, encryption, CloudTrail audit |
| Performance | 19 | Response time benchmarks (<5s SLA) |